# T2 — Docker: a API em um container

**Objetivo:** empacotar a API com tudo o que ela precisa para rodar. A mesma imagem roda no seu Codespace,
no GitHub Actions e em qualquer servidor.

> ⚠️ Antes de começar, pare o `uvicorn` da T1 (**Ctrl + C** no terminal dele): o container vai usar a porta 8000.

```
Dockerfile ──► docker build ──► imagem (bike-api) ──► docker run ──► container (API no ar)
 (a receita)                    (o pacote, imutável)                 (a imagem em execução)
```

| Conceito | O que é |
|---|---|
| **Dockerfile** | a receita: sistema base, dependências, código e o comando de partida |
| **Imagem** | o pacote pronto, somente leitura, feito em camadas |
| **Container** | uma execução da imagem; pode haver vários da mesma imagem |
| **Registry** | onde as imagens ficam guardadas (Docker Hub, GitHub Container Registry) |

**Comandos principais**

| Comando | O que faz |
|---|---|
| `docker build -t bike-api .` | constrói a imagem `bike-api` a partir do `Dockerfile` da pasta atual |
| `docker images` | lista as imagens guardadas |
| `docker run -d --name bike-api -p 8000:8000 --env-file .env bike-api` | cria e liga um container a partir da imagem |
| `docker ps` · `docker ps -a` | lista os containers ligados · todos, inclusive os parados |
| `docker logs bike-api` | mostra o que a API escreveu desde que ligou |
| `docker stop bike-api` · `docker start bike-api` | desliga · religa o container (ele continua existindo) |
| `docker restart bike-api` | desliga e liga de novo: a API carrega o `@champion` outra vez |
| `docker exec bike-api <comando>` | roda um comando dentro do container |
| `docker rm -f bike-api` | apaga o container (a imagem continua) |
| `docker system df` | quanto espaço as imagens e os containers ocupam |

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
_ = load_dotenv(".env", override=True)   # endereço e credenciais do MLflow (T0)

---
# Parte 1 · Juntos

## 1. As dependências da imagem

A imagem leva só o que a API usa para responder. O `requirements.txt` do projeto tem DVC, Jupyter e o MLflow
completo, que servem para treinar. A API precisa apenas **carregar** o modelo: para isso basta o `mlflow-skinny`.

In [2]:
%%writefile requirements-api.txt
fastapi==0.162.4
uvicorn==0.54.0
mlflow-skinny==3.17.0
scikit-learn==1.9.1
pandas==3.0.6
cloudpickle==3.1.2
python-dotenv==1.2.4

Writing requirements-api.txt


## 2. O Dockerfile

Cada instrução gera uma **camada**. O Docker guarda as camadas em cache e só refaz a partir da primeira que
mudou. Por isso as dependências (que mudam pouco) vêm antes do código (que muda sempre).

> 📁 Este `Dockerfile` fica na raiz e monta a **imagem da API**. O arquivo da pasta `.devcontainer`
> (`codespace.Dockerfile`) é outra coisa: monta o próprio Codespace, o ambiente em que você trabalha.

In [3]:
%%writefile Dockerfile
# Imagem base: Python 3.12 em um Linux enxuto
FROM python:3.12-slim

WORKDIR /app

# 1) Dependências primeiro: esta camada só é refeita quando o requirements-api.txt muda
COPY requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt

# 2) Código da API por último: é o que mais muda
COPY api/ api/

EXPOSE 8000
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


## 3. O `.dockerignore`

O `docker build` envia a pasta inteira para o Docker (o *contexto*). O `.dockerignore` deixa de fora o que não
deve ir: dados, histórico do Git e, principalmente, o `.env`.

In [4]:
%%writefile .dockerignore
.env
.git
.dvc
data
models
reports
mlartifacts
mlflow.db
aula*
desafio*
**/__pycache__

Writing .dockerignore


## 4. Construir a imagem

Na primeira vez demora alguns minutos (baixa a imagem base e instala as dependências).

In [5]:
%%bash
docker build -t bike-api .
docker images bike-api

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 448B done
#1 DONE 0.1s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 0.7s

#4 [internal] load .dockerignore
#4 transferring context: 127B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 3.50kB done
#5 DONE 0.1s

#6 [1/5] FROM docker.io/library/python:3.12-slim@sha256:5b5fe805d367f8f5a359ded6ab2652b7f9a206453ee6ee789c0045ba6ec15016
#6 resolve docker.io/library/python:3.12-slim@sha256:5b5fe805d367f8f5a359ded6ab2652b7f9a206453ee6ee789c0045ba6ec15016 0.1s done
#6 DONE 0.2s

#6 [1/5] FROM docker.io/library/python:3.12-slim@sha256:5b5fe805d367f8f5a359ded6ab2652b7f9a206453ee6ee789c0045ba6ec15016
#6 sha256:286d8c291daa64125703b22c3d009462bcae0b630eaa16ad

IMAGE             ID             DISK USAGE   CONTENT SIZE   EXTRA
bike-api:latest   268ab2692016        794MB          177MB        


In [7]:
%%bash
#docker images
docker ps

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


## 5. Rodar o container

- `-d` roda em segundo plano; `--name` dá um nome ao container.
- `-p 8000:8000` liga a porta 8000 do Codespace à porta 8000 do container.
- `--env-file .env` entrega as credenciais **na hora de rodar**. Elas não estão dentro da imagem.
- No `docker logs`, um `WARNING` sobre versões (numpy, psutil, rich) é esperado: o modelo anotou as versões do
  ambiente em que foi treinado (o Codespace), e a imagem leva só o necessário para rodar. A previsão funciona.

In [13]:
%%bash
docker run -d --name bike-api -p 8000:8000 --env-file .env bike-api
sleep 15
docker ps
docker logs bike-api

docker: Error response from daemon: Conflict. The container name "/bike-api" is already in use by container "50afe88da610a9d61fd1f41e3781d3ba7005c7a72cf86d6b71958f1c722a368f". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


Agora quem responde na porta 8000 é o **container**, e não o `uvicorn` da T1 (que você parou). A resposta tem de
ser a mesma da T1: versão 1 e 842 aluguéis para a hora do exemplo.

In [10]:
%%bash
curl -s localhost:8000/health
echo
curl -s -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json

{"status":"ok","model":"bike-demand","model_version":"6"}
{"cnt":852,"model_version":"6"}

De onde veio o modelo? A imagem leva só o código: o `.dockerignore` deixou `models/` de fora. O container
recebeu o endereço do DagsHub pelo `--env-file` e baixou o `@champion` de lá ao ligar. Confira: dentro do
container há só `api` e `requirements-api.txt`, e o `MLFLOW_TRACKING_URI` aponta para o DagsHub.

In [16]:
%%bash
docker exec bike-api ls /app
docker exec bike-api printenv MLFLOW_TRACKING_URI

Error response from daemon: container 50afe88da610a9d61fd1f41e3781d3ba7005c7a72cf86d6b71958f1c722a368f is not running
Error response from daemon: container 50afe88da610a9d61fd1f41e3781d3ba7005c7a72cf86d6b71958f1c722a368f is not running


CalledProcessError: Command 'b'docker exec bike-api ls /app\ndocker exec bike-api printenv MLFLOW_TRACKING_URI\n'' returned non-zero exit status 1.

## 6. A imagem não tem modelo nem senha

A célula liga um segundo container **sem o `--env-file`**. Sem as variáveis do DagsHub, o MLflow usa o endereço
padrão, local (o `sqlite:///mlflow.db` que aparece no erro), onde não há modelo nenhum: a API desiste e o
container termina com `Application startup failed`. É o comportamento certo: configuração e segredos entram na
execução, não na imagem.

- `--rm` apaga o container assim que ele termina (é só um teste).
- `2>&1 | tail -3` junta as mensagens de erro à saída e mostra só as 3 últimas linhas.

In [14]:
%%bash
docker run --rm bike-api 2>&1 | tail -3

mlflow.tracking.registry.UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'sqlite:////app/mlflow.db' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.

ERROR:    Application startup failed. Exiting.


## 7. Camadas e cache

Mude qualquer coisa em `api/main.py` (um comentário, por exemplo) e construa de novo: só a última camada é refeita.

In [15]:
%%bash
echo "# build de teste" >> api/main.py
docker build -t bike-api . 2>&1 | grep -E "CACHED|COPY|RUN"
git checkout api/main.py
docker history bike-api --format "table {{.CreatedBy}}\t{{.Size}}" | head -8

#7 CACHED
#8 [3/5] COPY requirements-api.txt .
#8 CACHED
#9 [4/5] RUN pip install --no-cache-dir -r requirements-api.txt
#9 CACHED
#10 [5/5] COPY api/ api/


Updated 1 path from the index


CREATED BY                                      SIZE
CMD ["uvicorn" "api.main:app" "--host" "0.0.…   0B
EXPOSE 8000/tcp                                 0B
COPY api/ api/ # buildkit                       20.5kB
RUN /bin/sh -c pip install --no-cache-dir -r…   483MB
COPY requirements-api.txt . # buildkit          12.3kB
WORKDIR /app                                    8.19kB
CMD ["python3"]                                 0B


---
# Parte 2 · Sozinhos

Na Aula 1 vimos que promover é **mover um ponteiro**. Agora há um consumidor de verdade no ar: o container.
Você vai promover um modelo novo e depois voltar ao anterior, **sem construir a imagem de novo**.

## 1. Treine uma versão nova

A célula reduz o número de árvores no `params.yaml` (de 300 para 5). Rode o pipeline (como na T1 da Aula 1).

In [24]:
%%bash
# sed -i 's/max_iter: 300/max_iter: 5/' params.yaml
# TODO: mostre a diferença de parâmetros e rode o pipeline

sed -i 's/max_iter: 300/max_iter: 5/' params.yaml
dvc params diff
dvc repro

Path         Param           HEAD    workspace
params.yaml  train.max_iter  300     5
'data/raw/bike.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'split' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Stage 'evaluate' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Stage 'plot' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Running stage 'compare':
> python desafio01/compare_models.py
            versao     rmse      mae     r2
champion       6.0   62.894   41.506  0.848
challenger     7.0  126.346  102.385  0.386

Teste: 2012-12-04 a 2012-12-31 (670 linhas)
Challenger NÃO venceu: RMSE 126.3 vs 62.9 (champion)
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


## 2. Promova a versão nova (como no passo 5 da T3 da Aula 1)

Alguém promoveu a versão nova **sem validar**. Faça o mesmo, de propósito: mova o `@champion` para ela.

In [18]:
import json

from mlflow import MlflowClient
from bike import MODEL_NAME

client = MlflowClient()
nova = json.load(open("models/run.json"))["model_version"]
anterior = client.get_model_version_by_alias(MODEL_NAME, "champion").version

# TODO: grave a tag previous_champion na versão nova e mova o alias champion para ela

client.set_model_version_tag(MODEL_NAME, str(nova), "previous_champion", str(anterior))
client.set_registered_model_alias(MODEL_NAME, "champion", str(nova))

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. A API já está usando o modelo novo?

Consulte o `/health` e peça a previsão do exemplo. Depois reinicie o container e repita.

In [19]:
%%bash
curl -s localhost:8000/health
echo
curl -s -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json

{"status":"ok","model":"bike-demand","model_version":"6"}
{"cnt":852,"model_version":"6"}

In [20]:
%%bash
# TODO: reinicie o container (docker restart bike-api), espere uns 15 segundos (sleep 15) e repita as duas chamadas
docker restart bike-api
sleep 15
curl -s localhost:8000/health
echo
curl -s -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json

Error response from daemon: Cannot restart container bike-api: failed to set up container networking: driver failed programming external connectivity on endpoint bike-api (a6f90171df5bdf5c8f7cc452ce8a6a137e8618bfe2bfa0c1c4b7997a7ffb5ade): failed to bind host port 0.0.0.0:8000/tcp: address already in use


{"status":"ok","model":"bike-demand","model_version":"6"}
{"cnt":852,"model_version":"6"}

> 💬 Compare as duas previsões para a mesma hora. A API não deu erro nenhum: respondeu 200, com um número.

## 4. Desfaça o experimento e guarde

O `@champion` volta para a versão anterior (a T3 precisa do modelo bom no ar) e o container é reiniciado para
carregá-lo. Depois, o `params.yaml` volta ao que era e o `dvc repro` restaura o modelo anterior do cache, sem
treinar de novo.

In [21]:
atual = client.get_model_version_by_alias(MODEL_NAME, "champion")
anterior = atual.tags["previous_champion"]

client.set_registered_model_alias(MODEL_NAME, "champion", anterior)
print(f"champion: versão {atual.version} -> {anterior}")

champion: versão 7 -> 6


In [22]:
%%bash
docker restart bike-api
sleep 15
curl -s localhost:8000/health
echo
curl -s -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json

Error response from daemon: Cannot restart container bike-api: failed to set up container networking: driver failed programming external connectivity on endpoint bike-api (03e258c25f259674fc613963d1ae97d63086d8a84133e3b5ad30340ec80963f7): failed to bind host port 0.0.0.0:8000/tcp: address already in use


{"status":"ok","model":"bike-demand","model_version":"6"}
{"cnt":852,"model_version":"6"}

In [23]:
%%bash
git checkout params.yaml
dvc repro
# TODO: faça o commit do Dockerfile, do .dockerignore e do requirements-api.txt, e o push
git add Dockerfile .dockerignore requirements-api.txt
git commit -m "Adiciona configuracao Docker da API"
git push

Updated 1 path from the index


'data/raw/bike.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'split' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Stage 'evaluate' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Stage 'plot' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
[aula_02 ebce130] Adiciona configuracao Docker da API
 3 files changed, 32 insertions(+)
 create mode 100644 .dockerignore
 create mode 100644 Dockerfile
 create mode 100644 requirements-api.txt


To https://github.com/anasegatelli/mlops-facens
   91b34af..ebce130  aula_02 -> aula_02
